# TMFT Experiment Notebook

Upload the entire `tmft_project/` directory to Vessel, then open this notebook from inside that directory. This notebook depends on `src/`, `configs/config.yaml`, `requirements.txt`, and `main.py`; uploading only the `.ipynb` file is not enough.

## 0. Expected Project Structure

The working directory should contain:

- `configs/config.yaml`
- `src/train.py`
- `src/masking.py`
- `src/evaluate_pii.py`
- `requirements.txt`
- `main.py`

In [ ]:
from pathlib import Path
import os, sys, json, platform

PROJECT_ROOT = Path.cwd()
print('PROJECT_ROOT =', PROJECT_ROOT)
print('Python =', sys.version)
print('Platform =', platform.platform())

required_paths = [
    'configs/config.yaml',
    'src/train.py',
    'src/masking.py',
    'src/evaluate_pii.py',
    'requirements.txt',
    'main.py',
]
missing = [p for p in required_paths if not (PROJECT_ROOT / p).exists()]
if missing:
    raise FileNotFoundError('You are not inside the full tmft_project directory. Missing: ' + ', '.join(missing))
print('Project structure OK')

## 1. Install Dependencies

Run this once in a fresh Vessel workspace. Restart the kernel after installation if imports fail.

In [ ]:
%pip install -U pip
%pip install -r requirements.txt
!python -m spacy download en_core_web_sm

## 2. Import Code and Check GPU

In [ ]:
import os
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')

import torch
import pandas as pd
from transformers import AutoModelForCausalLM

from src.train import METHODS, load_config, load_text_dataset, load_tokenizer, train_model, upload_to_huggingface
from src.evaluate_pii import evaluate_pii, load_pii_eval_set

print('Available methods:', METHODS)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Load and Edit Configuration

Use the smoke-test values first. For the final run, increase `max_train_samples` and `num_epochs`.

In [ ]:
config = load_config('configs/config.yaml')

# Smoke-test settings. Change these for the final experiment.
config['max_train_samples'] = 8
config['num_epochs'] = 0.01
config['batch_size'] = 1
config['gradient_accumulation_steps'] = 1
config['save_steps'] = 20
config['logging_steps'] = 1
config['output_dir'] = './results'

config

## 4. Load Dataset

In [ ]:
dataset = load_text_dataset(config)
print(dataset)
print(dataset[0][config['text_column']][:500])

## 5. Smoke Test One Method

Run one small method first to verify downloads, tokenizer, LoRA, masking, and Trainer are working.

In [ ]:
SMOKE_METHOD = 'tmft_ner'  # baseline, rmft, tmft_ner, tmft_mia, tmft_combined
trainer, tokenizer, output_dir = train_model(config, method=SMOKE_METHOD, train_dataset=dataset)
print('Saved smoke-test model to:', output_dir)

## 6. Final Experiment Settings

After the smoke test passes, edit these values for the real run.

In [ ]:
final_config = load_config('configs/config.yaml')

# Suggested starting point. Adjust based on Vessel GPU memory and deadline.
final_config['max_train_samples'] = 1000
final_config['num_epochs'] = 1
final_config['batch_size'] = 1
final_config['gradient_accumulation_steps'] = 4
final_config['save_steps'] = 250
final_config['logging_steps'] = 10
final_config['output_dir'] = './results'

final_config

## 7. Train All Methods

This is the main experiment cell. It may take a while. If time is limited, start with `['baseline', 'rmft', 'tmft_ner']`.

In [ ]:
methods_to_run = ['baseline', 'rmft', 'tmft_ner', 'tmft_mia', 'tmft_combined']
train_outputs = {}

final_dataset = load_text_dataset(final_config)
for method in methods_to_run:
    print('\n===== TRAIN:', method, '=====')
    trainer, tokenizer, output_dir = train_model(final_config, method=method, train_dataset=final_dataset)
    train_outputs[method] = str(output_dir)

train_outputs

## 8. Prepare Evaluation Set

Create `data/pii_eval.json` with records like:

```json
[{"prefix": "...", "ground_truth_target": "..."}]
```

The evaluation cell below expects that file.

In [ ]:
eval_path = Path('data/pii_eval.json')
if not eval_path.exists():
    example_eval = [
        {
            'prefix': 'Dear team, please contact',
            'ground_truth_target': 'FILL_WITH_REAL_PII_TARGET'
        }
    ]
    eval_path.write_text(json.dumps(example_eval, indent=2), encoding='utf-8')
    print('Created template:', eval_path)
    print('Edit this file with real prefix/target pairs before using metrics in the report.')
else:
    print('Evaluation file exists:', eval_path)

## 9. Evaluate PII Extraction Rate

In [ ]:
eval_set = load_pii_eval_set(eval_path)
eval_results = []

for method, model_dir in train_outputs.items():
    print('\n===== EVAL:', method, '=====')
    model = AutoModelForCausalLM.from_pretrained(model_dir)
    tokenizer = load_tokenizer(model_dir)
    if torch.cuda.is_available():
        model = model.cuda()
    metrics = evaluate_pii(model, tokenizer, eval_set)
    metrics['method'] = method
    metrics['model_dir'] = model_dir
    eval_results.append(metrics)

results_df = pd.DataFrame(eval_results)
results_df

## 10. Save Result Table for the Presentation

In [ ]:
Path('results').mkdir(exist_ok=True)
results_csv = Path('results/tmft_eval_results.csv')
results_df.to_csv(results_csv, index=False)
print('Saved:', results_csv)
results_df

## 11. Hugging Face Upload

Run `huggingface-cli login` in a Vessel terminal first, or set `HF_TOKEN` according to Vessel's secret/environment variable UI.

In [ ]:
HF_REPO_PREFIX = 'YOUR_USERNAME/tmft-pythia-160m'
method_to_upload = 'tmft_combined'

model_dir = train_outputs.get(method_to_upload, f"results/{method_to_upload}")
repo_id = f"{HF_REPO_PREFIX}-{method_to_upload}"

# Uncomment after login and after checking repo_id.
# upload_to_huggingface(model_dir, repo_id=repo_id, private=True)
print('Ready to upload:', model_dir, '->', repo_id)